# Retrieval Augmented Generation (RAG) with Langchain
*Using IBM Granite Models*

## In this notebook
This notebook contains instructions for performing Retrieval Augumented Generation (RAG). RAG is an architectural pattern that can be used to augment the performance of language models by recalling factual information from a knowledge base, and adding that information to the model query. The most common approach in RAG is to create dense vector representations of the knowledge base in order to retrieve text chunks that are semantically similar to a given user query.

RAG use cases include:
- Customer service: Answering questions about a product or service using facts from the product documentation.
- Domain knowledge: Exploring a specialized domain (e.g., finance) using facts from papers or articles in the knowledge base.
- News chat: Chatting about current events by calling up relevant recent news articles.

In its simplest form, RAG requires 3 steps:

- Initial setup:
  - Index knowledge-base passages for efficient retrieval. In this recipe, we take embeddings of the passages and store them in a vector database.
- Upon each user query:
  - Retrieve relevant passages from the database. In this recipe, we use an embedding of the query to retrieve semantically similar passages.
  - Generate a response by feeding retrieved passage into a large language model, along with the user query.

## Setting up the environment

### Python version

Ensure you are running Python 3.10 or 3.11.

In [1]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 12), "Use Python 3.10 or 3.11 to run this notebook."

### Install dependencies

Granite Kitchen comes with a bundle of dependencies that are required for notebooks. See the list of packages in its [`setup.py`](https://github.com/ibm-granite-community/granite-kitchen/blob/main/setup.py). 

In [3]:
! pip install "git+https://github.com/alanbraz/granite-kitchen.git" "langchain-huggingface" "langchain-milvus" "wget"

  Cloning https://github.com/alanbraz/granite-kitchen.git to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-9trub0f6
  Running command git clone --filter=blob:none --quiet https://github.com/alanbraz/granite-kitchen.git /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-9trub0f6
  Resolved https://github.com/alanbraz/granite-kitchen.git to commit b5f7e403f3fffa7f6391a005655e70670c415f1f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached langchain_huggingface-0.1.2-py3-none-any.whl.metadata (1.3 kB)
  Using cached langchain_milvus-0.1.7-py3-none-any.whl.metadata (1.9 kB)
  Cloning https://github.com/ibm-granite-community/utils to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-install-x3y1pfff/ibm-granite-community-utils_224bbf67cad44470b9f4947f2f082aca
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-gra

### Serving the Granite AI model


This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://github.com/ollama/ollama) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

During the pre-work, you may have either started a local Ollama server on your computer, or setup Replicate access and obtained an [API token](https://replicate.com/account/api-tokens).

## Selecting System Components

### Choose your Embeddings Model

Specify the model to use for generating embedding vectors from text.

To use a model from a provider other than Huggingface, replace this code cell with one from [this Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

### Choose your Vector Database

Specify the database to use for storing and retrieving embedding vectors.

To connect to a vector database other than Milvus substitute this code cell with one from [this Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [3]:
from langchain_milvus import Milvus
import tempfile

db_file = tempfile.NamedTemporaryFile(prefix="milvus_", suffix=".db", delete=False).name
print(f"The vector database will be saved to {db_file}")

vector_db = Milvus(embedding_function=embeddings_model, connection_args={"uri": db_file}, auto_id=True)

The vector database will be saved to /var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/milvus_2j44ep54.db


## Select your model


Select a Granite model to use. Here we use a Langchain client to connect to the model. If there is a locally accessible Ollama server, we use an Ollama client to access the model. Otherwise, we use a Replicate client to access the model.

When using Replicate, if the `REPLICATE_API_TOKEN` environment variable is not set, or a `REPLICATE_API_TOKEN` Colab secret is not set, then the notebook will ask for your [Replicate API token](https://replicate.com/account/api-tokens) in a dialog box.

In [5]:
import os
import requests
from langchain_ollama.llms import OllamaLLM
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import set_env_var

try: # Look for a locally accessible Ollama server for the model
    response = requests.get(os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434"))
    model = OllamaLLM(model="granite3-dense:8b")
except Exception: # Use Replicate for the model
    set_env_var("REPLICATE_API_TOKEN")
    model = Replicate(model="ibm-granite/granite-3.0-8b-instruct")


## Building the Vector Database

In this example, we take the State of the Union speech text, split it into chunks, derive embedding vectors using the embedding model, and load it into the vector database for querying.

### Download the document

Here we use President Biden's State of the Union address from March 1, 2022.

In [16]:
import wget

filename = 'state_of_the_union.txt'
url = 'https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/foundation_models/state_of_the_union.txt'

if not os.path.isfile(filename):
  wget.download(url, out=filename)

### Split the document into chunks

Split the document into text segments that can fit into the model's context window.

In [18]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

Created a chunk of size 304, which is longer than the specified 300
Created a chunk of size 332, which is longer than the specified 300
Created a chunk of size 325, which is longer than the specified 300


### Populate the vector database

NOTE: Population of the vector database may take over a minute depending on your embedding model and service.

In [19]:
vector_db.add_documents(texts)

[454420933529567234,
 454420933529567235,
 454420933529567236,
 454420933529567237,
 454420933529567238,
 454420933529567239,
 454420933529567240,
 454420933529567241,
 454420933529567242,
 454420933529567243,
 454420933529567244,
 454420933529567245,
 454420933529567246,
 454420933529567247,
 454420933529567248,
 454420933529567249,
 454420933529567250,
 454420933529567251,
 454420933529567252,
 454420933529567253,
 454420933529567254,
 454420933529567255,
 454420933529567256,
 454420933529567257,
 454420933529567258,
 454420933529567259,
 454420933529567260,
 454420933529567261,
 454420933529567262,
 454420933529567263,
 454420933529567264,
 454420933529567265,
 454420933529567266,
 454420933529567267,
 454420933529567268,
 454420933529567269,
 454420933529567270,
 454420933529567271,
 454420933529567272,
 454420933529567273,
 454420933529567274,
 454420933529567275,
 454420933529567276,
 454420933529567277,
 454420933529567278,
 454420933529567279,
 454420933529567280,
 454420933529

## Querying the Vector Database

### Conduct a similarity search

Search the database for similar documents by proximity of the embedded vector in vector space.

In [20]:
query = "What did the president say about Ketanji Brown Jackson"
docs = vector_db.similarity_search(query)
print(docs[0].page_content)

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


## Answering Questions

### Automate the RAG pipeline

Build a RAG chain with the model and the document retriever. See the [Granite Prompting Guide](https://www.ibm.com/granite/docs/models/granite/#prompt-anatomy) for information on the prompt template.

In [22]:
from langchain.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create a prompt template for question-answering with the retrieved context.
prompt_template = """\
<|start_of_role|>user<|end_of_role|>Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {input}<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>"""

# Assemble the retrieval-augmented generation chain.
qa_chain_prompt = PromptTemplate.from_template(prompt_template)
combine_docs_chain = create_stuff_documents_chain(model, qa_chain_prompt)
rag_chain = create_retrieval_chain(vector_db.as_retriever(), combine_docs_chain)

### Generate a retrieval-augmented response to a question

Use the RAG chain to process the query. 

In [23]:
output = rag_chain.invoke({"input": "What did the president say about Ketanji Brown Jackson?"})

print(output['answer'])

The president nominated Ketanji Brown Jackson, a Circuit Court of Appeals Judge, to continue Justice Breyer's legacy of excellence.
